<a href="https://colab.research.google.com/github/adriari4/lab-vertical-chat/blob/main/lab-vertical-chat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vertical Chat
A sample how to build a chat for small business using:

* GPT 35
* Panel
* OpenAI


This is just a simple sample to start to understand how the OpenAI API works, and how to create Prompts. It Is really far from beign a complete solution.
We are going to introduce some interesting points:

* The roles in a conversation.
* How is the conversations’ memory preserved?

Deeper explanations in the article: [Create Your First Chatbot Using GPT 3.5, OpenAI, Python and Panel.](https://medium.com/towards-artificial-intelligence/create-your-first-chatbot-using-gpt-3-5-openai-python-and-panel-7ec180b9d7f2)

In [1]:
#if you need an API Key from OpenAI
#https://platform.openai.com/account/api-keys

from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [3]:
# Step 1: install dependencies (only first time)
!pip install openai python-dotenv

# Step 2: import and enter your key securely
from getpass import getpass
from openai import OpenAI

OPENAI_API_KEY = getpass("🔑 Enter your OpenAI API key: ")

# Step 3: initialize client
client = OpenAI(api_key=OPENAI_API_KEY)

# Test it
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": "Say hi"}]
)
print(response.choices[0].message.content)


🔑 Enter your OpenAI API key: ··········
Hi there! How can I assist you today?


In [4]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def continue_conversation(messages, temperature=0):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        temperature=temperature,
    )
    #print(str(response.choices[0].message["content"]))
    return response.choices[0].message.content

In [5]:
def add_prompts_conversation(_):
    #Get the value introduced by the user
    prompt = client_prompt.value_input
    client_prompt.value = ''

    #Append to the context the User prompt.
    context.append({'role':'user', 'content':f"{prompt}"})

    #Get the response.
    response = continue_conversation(context)

    #Add the response to the context.
    context.append({'role':'assistant', 'content':f"{response}"})

    #Update the panels to show the conversation.
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600)))

    return pn.Column(*panels)

In [6]:
#Creating the prompt
#read and understand it.
import panel as pn  # GUI

context = [ {'role':'system', 'content':"""
Act as an OrderBot, you work collecting orders in a delivery only fast food restaurant called
My Dear Frankfurt. \
First welcome the customer, in a very friendly way, then collects the order. \
You wait to collect the entire order, beverages included \
then summarize it and check for a final \
time if everything is ok or the customer wants to add anything else. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very friendly style. \
The menu includes \
burger  12.95, 10.00, 7.00 \
frankfurt   10.95, 9.25, 6.50 \
sandwich   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
martra sausage 3.00 \
canadian bacon 3.50 \
romesco sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
vichy catalan 5.00 \
"""} ]

#Creating the panel.
pn.extension()

panels = []

client_prompt = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="talk")

interactive_conversation = pn.bind(add_prompts_conversation, button_conversation)

dashboard = pn.Column(
    client_prompt,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True),
)

dashboard

/tmp/ipython-input-3485984612.py:36: UserWarning: Using Panel interactively in Colab notebooks requires the jupyter_bokeh package to be installed. Install it with:

    !pip install jupyter_bokeh

and try again.
  pn.extension()


Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(name='talk')
    [2] ParamFunction(function, _pane=Column, defer_load=False, loading_indicator=True)

# Exercise
 - Complete the prompts similar to what we did in class.
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

# **EmotiBot – The Teen Emotion Helper**


*   Goal: Practice “Give the model time to think” and “Check conditions”.

*   Idea: The bot detects the user’s emotion and responds empathetically.

Prompting Principles Used
Principle	How it applies
1. Clear and specific instructions	We tell the model its exact role and structure of the answer.
2. Give the model time to think	We make it analyze first, then respond empathetically.
Tactic: Use delimiters	Triple backticks for user input.
Tactic: Ask for structured output	JSON format: emotion, description, advice.
Tactic: Work out reasoning first	Model identifies the emotion before giving advice.




In [13]:
def get_completion(prompt, model="gpt-3.5-turbo"):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.9,  # a bit of creativity for empathy
    )
    return response.choices[0].message.content

In [9]:
#Creating the prompt
#read and understand it.
import panel as pn  # GUI

context = [ {'role':'system', 'content':"""
You are EmotiBot, a friendly AI that helps teenagers understand their emotions.

Steps:
1. Identify the main emotion(s) in the message delimited by triple backticks.
2. Describe what that emotion might mean.
3. Suggest one healthy coping action.
4. Output everything in JSON format with keys: emotion, description, advice.

User message:
```{user_message}```
"""} ]

#Creating the panel.
pn.extension()

panels = []

client_prompt = pn.widgets.TextInput(value="💬 EmotiBot is here to listen. Type 'quit' to stop.\n", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="talk")

interactive_conversation = pn.bind(add_prompts_conversation, button_conversation)

dashboard = pn.Column(
    client_prompt,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True),
)

dashboard

/tmp/ipython-input-1958573258.py:19: UserWarning: Using Panel interactively in Colab notebooks requires the jupyter_bokeh package to be installed. Install it with:

    !pip install jupyter_bokeh

and try again.
  pn.extension()


Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(name='talk')
    [2] ParamFunction(function, _pane=Column, defer_load=False, loading_indicator=True)

In [14]:
# 🔁 Restart chatbot conversation and environment
panels = []           # clear previous messages
client_prompt.value = ""  # clear text box

print("✅ Chatbot restarted. You can now load or start a new one (e.g., StudyBot, FocusBot, etc.)")


✅ Chatbot restarted. You can now load or start a new one (e.g., StudyBot, FocusBot, etc.)


In [15]:
# Panel setup
pn.extension()

# Function to process conversation
def add_prompts_conversation(event=None):
    user_message = client_prompt.value
    prompt = f"""
You are EmotiBot, a friendly AI that helps teenagers understand their emotions.

Steps:
1. Identify the main emotion(s) in the message delimited by triple backticks.
2. Describe what that emotion might mean.
3. Suggest one healthy coping action.
4. Output everything in JSON format with keys: emotion, description, advice.

User message:
```{user_message}```
"""
    response = get_completion(prompt)
    panels.append(pn.Row("🧑‍🎓 You:", user_message))
    panels.append(pn.Row("🤖 EmotiBot:", pn.pane.Markdown(response, width=600)))
    return pn.Column(*panels)

# Widgets
client_prompt = pn.widgets.TextInput(placeholder='Write how you feel…')
button_conversation = pn.widgets.Button(name="Talk", button_type='primary')

# Bind button to action
interactive_conversation = pn.bind(add_prompts_conversation, button_conversation)

# Dashboard layout
dashboard = pn.Column(
    pn.pane.Markdown("## 💬 EmotiBot – Teen Emotion Helper\nTell me how you feel 👇", width=600),
    client_prompt,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True),
)

dashboard

/tmp/ipython-input-2740399291.py:2: UserWarning: Using Panel interactively in Colab notebooks requires the jupyter_bokeh package to be installed. Install it with:

    !pip install jupyter_bokeh

and try again.
  pn.extension()


Column
    [0] Markdown(str, width=600)
    [1] TextInput(placeholder='Write how you feel…')
    [2] Row
        [0] Button(button_type='primary', name='Talk')
    [3] ParamFunction(function, _pane=Column, defer_load=False, loading_indicator=True)

# CHATBOT: HELPER TO FIX HOME S ISSUES

In [40]:
!pip install jupyter_bokeh


In [41]:
# 🔄 Clear all previous chatbot sessions (EmotiBot etc.)
try:
    panels = []
except NameError:
    pass

try:
    client_prompt.value = ""
except NameError:
    pass

# Remove any old function definitions or context
for name in ["add_prompts_conversation", "context"]:
    if name in globals():
        del globals()[name]

print("✅ Chatbot environment fully reset. Ready for HomeFixBot.")



✅ Chatbot environment fully reset. Ready for HomeFixBot.


In [42]:
def get_completion(prompt, model="gpt-3.5-turbo"):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.3,  # LESSER CREATIVY
    )
    return response.choices[0].message.content

In [44]:
def add_prompts_conversation(_):
    #Get the value introduced by the user
    prompt = client_prompt.value_input
    client_prompt.value = ''

    #Append to the context the User prompt.
    context.append({'role':'user', 'content':f"{prompt}"})

    #Get the response.
    response = continue_conversation(context)

    #Add the response to the context.
    context.append({'role':'assistant', 'content':f"{response}"})

    #Update the panels to show the conversation.
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600)))

    return pn.Column(*panels)

In [46]:
#Creating the prompt
#read and understand it.
import panel as pn  # GUI

context = [ {'role':'system', 'content':"""
You are HomeFixBot, A chatbot that helps users identify, explain, and suggest solutions for everyday home issues —
like plumbing, electricity, furniture, Wi-Fi, cleaning, or maintenance.
Steps:
1. Identify the main problem in the message delimited by triple backticks.
2. Describe what that probelm might mean.
3. Suggest the steps to fix the issue. Such as: the material to buy, the steps to fix
4. Output everything in just human-readable text.



Follow this format:
1️⃣ Greeting or short empathetic reaction
2️⃣ Quick explanation of what might be happening
3️⃣ Simple step or advice
4️⃣ Optional emoji or short encouragement line

Example:
User: "My washing machine is leaking water."
HomeFixBot:
"Hey! 👋 Sounds like your washing machine hose might be loose.
Try tightening it gently at the back — make sure the water is off first 💧
If it still leaks, call a technician just to be safe 👍"



```{user_message}```
"""} ]

#Creating the panel.
pn.extension()

panels = []

client_prompt = pn.widgets.TextInput(value="💬 HomeFixBot is here to listen. Type 'quit' to stop.\n", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="talk")

interactive_conversation = pn.bind(add_prompts_conversation, button_conversation)

dashboard = pn.Column(
    client_prompt,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True),
)

dashboard

Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(name='talk')
    [2] ParamFunction(function, _pane=Column, defer_load=False, loading_indicator=True)

# SPORT RESULT CHATBOT



In [54]:
# 🔄 Clear all previous chatbot sessions (EmotiBot etc.)
try:
    panels = []
except NameError:
    pass

try:
    client_prompt.value = ""
except NameError:
    pass

# Remove any old function definitions or context
for name in ["add_prompts_conversation", "context"]:
    if name in globals():
        del globals()[name]

print("✅ Chatbot environment fully reset. Ready for HomeFixBot.")

✅ Chatbot environment fully reset. Ready for HomeFixBot.


In [55]:
def get_completion(prompt, model="gpt-3.5-turbo"):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,  # LESSER CREATIVY
    )
    return response.choices[0].message.content

def add_prompts_conversation(event=None):
    user_message = client_prompt.value
    client_prompt.value = '' # Clear input box after sending

    if user_message.lower() == 'quit':
        panels.append(pn.Row("🤖 ResultMate:", pn.pane.Markdown("Goodbye! Let me know if you have more sports questions later.", width=600)))
        client_prompt.disabled = True
        button_conversation.disabled = True
    else:
        prompt = f"""
You are ResultMate, a sports-fantastic friend who shares latest results and upcoming fixtures in a casual, fun way.

Tactics:
1. Be clear and specific: ask what sport and teams/leagues the user means.
2. Think before answering: check the user’s query, clarify if needed.
3. Use delimiters ``` to isolate what the user typed.
4. Provide structured but friendly output:
   - sport: …
   - game/result: …
   - if upcoming: next match and date/time
   - small comment line (fun fact or emoji).
5. Tone: short, conversational, like texting a friend (2-4 lines max).

Example:
<user>: ```What was the result of Real Madrid vs Barcelona in La Liga yesterday?```
<ResultMate>:
"sport: Football ⚽
game: Real Madrid 2-1 Barcelona
next up: none (this was yesterday)
cool fact: Barcelona shot 23 times today! 😉 ```{user_message}```
"""
        response = get_completion(prompt)
        panels.append(pn.Row("🧑‍🎓 You:", user_message))
        panels.append(pn.Row("🤖 ResultMate:", pn.pane.Markdown(response, width=600)))
    return pn.Column(*panels)

context = [ {'role':'system', 'content':"""
You are ResultMate, a sports-fantastic friend who shares latest results and upcoming fixtures in a casual, fun way.

Tactics:
1. Be clear and specific: ask what sport and teams/leagues the user means.
2. Think before answering: check the user’s query, clarify if needed.
3. Use delimiters ``` to isolate what the user typed.
4. Provide structured but friendly output:
   - sport: …
   - game/result: …
   - if upcoming: next match and date/time
   - small comment line (fun fact or emoji).
5. Tone: short, conversational, like texting a friend (2-4 lines max).

Example:
<user>: ```What was the result of Real Madrid vs Barcelona in La Liga yesterday?```
<ResultMate>:
"sport: Football ⚽
game: Real Madrid 2-1 Barcelona
next up: none (this was yesterday)
cool fact: Barcelona shot 23 times today! 😉 ```{user_message}```
"""} ]
#Creating the panel.
import panel as pn  # GUI


#Creating the panel.
pn.extension()

panels = []

client_prompt = pn.widgets.TextInput(value="💬 ResultMate is here! Ask me about sports results.\n", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="talk")

interactive_conversation = pn.bind(add_prompts_conversation, button_conversation)

dashboard = pn.Column(
    client_prompt,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True),
)

dashboard

Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(name='talk')
    [2] ParamFunction(function, _pane=Column, defer_load=False, loading_indicator=True)

- Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

What I learnt is that:


*  Last chatbot has some problems to identify the results of matchs
*  It missundertood sports.


*   I learned how each function works, how to connect with an API, and how to build the interface. Now I understand that real data requires integrating an external sports API

